In [ ]:
%sh curl -LsSf https://raw.githubusercontent.com/mlflow/mlflow/HEAD/dev/install-skinny.sh | sh -s e07bf239b9dcf95e23fce5e59dff091df17669bc

In [ ]:
%pip install -U "bitsandbytes>=0.46.1"

In [ ]:
%pip install -U "transformers==5.8.1" "trl==1.10.0"


In [ ]:
dbutils.library.restartPython()

In [ ]:
# Attach this notebook to Serverless GPU, 1xH100, AI v5 before running.
import os
import sys

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import logging
from pathlib import Path
from tempfile import mkdtemp

import mlflow
import torch
import transformers
import trl
import peft
from mlflow.tracking import MlflowClient
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

from drzavni_posao.transcription import MediaStore, MlflowTracker, Stage
from drzavni_posao.transcription.constants import ANONYMOUS_LABEL
from drzavni_posao.generation.dataset import (
    SFTDatasetBuilder,
    TranscriptQualityError,
    assign_folds,
    build_pair,
    select_splits,
)
from drzavni_posao.generation.inference import generate_episode

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

SOURCE_EXPERIMENT_NAME = "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
SOURCE_RUN_ID = "1c519fba673841fa9188addf9a720ae6"
PREMISE_EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/premise-generation"
)
PREMISE_RUN_ID = "4eadb1cda666464c90b6b01d71dbf574"
TRAINING_EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/screenplay-qlora"
)
WORKSPACE_ROOT = Path("/Workspace/Users/luka.erceg@databricks.com/drzavni_posao")

MODEL_KEY = "zora"  # Choose "qwen3" or "zora".
MODEL_NAMES = {
    "qwen3": "Qwen/Qwen3-8B",
    "zora": "sovasoft/zora-v1.12",
}
if MODEL_KEY not in MODEL_NAMES:
    raise ValueError(f"Unknown model key {MODEL_KEY!r}; choose from {sorted(MODEL_NAMES)}")
MODEL_NAME = MODEL_NAMES[MODEL_KEY]
RUN_NAME = f"{MODEL_KEY}-qlora-full-episode-v3"
MAX_LENGTH = 8192
MAX_NEW_TOKENS = 2048
GENERATION_DO_SAMPLE = True
GENERATION_TEMPERATURE = 0.8
GENERATION_TOP_P = 0.95
GENERATION_TOP_K = 0
GENERATION_REPETITION_PENALTY = 1.15
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-5
TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 0.3
SEED = 42

EXPECTED_TRANSFORMERS_VERSION = "5.8.1"
EXPECTED_TRL_VERSION = "1.10.0"
if transformers.__version__ != EXPECTED_TRANSFORMERS_VERSION:
    raise RuntimeError(
        f"Expected transformers {EXPECTED_TRANSFORMERS_VERSION}, "
        f"found {transformers.__version__}"
    )
if trl.__version__ != EXPECTED_TRL_VERSION:
    raise RuntimeError(f"Expected TRL {EXPECTED_TRL_VERSION}, found {trl.__version__}")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a GPU")
GPU_NAME = torch.cuda.get_device_name(0)
if "H100" not in GPU_NAME:
    raise RuntimeError(f"Expected an H100, found {GPU_NAME}")

set_seed(SEED)
print(
    {
        "gpu": GPU_NAME,
        "model_key": MODEL_KEY,
        "model_name": MODEL_NAME,
        "transformers": transformers.__version__,
        "trl": trl.__version__,
        "peft": peft.__version__,
    }
)

In [ ]:
source_tracker = MlflowTracker(SOURCE_EXPERIMENT_NAME)
states = source_tracker.episode_run_states(SOURCE_RUN_ID)

mlflow.set_experiment(PREMISE_EXPERIMENT_NAME)
traces = mlflow.search_traces(
    run_id=PREMISE_RUN_ID,
    max_results=1000,
    return_type="list",
)

premises = {}
empty_episode_keys = []
for trace in traces:
    if trace.info.tags.get("premise.complete") != "true":
        continue

    episode_key = trace.info.tags.get("episode_key")
    outputs = trace.data.spans[0].outputs if trace.data.spans else None
    premise = ((outputs or {}).get("premise") or "").strip()
    if not episode_key or not premise:
        if episode_key:
            empty_episode_keys.append(episode_key)
        continue
    if episode_key in premises:
        raise ValueError(f"Multiple valid premises found for episode {episode_key!r}")
    premises[episode_key] = premise

media = MediaStore(
    audio_dir=None,
    run_dir=Path("/tmp/screenplay-qlora"),
    catalog_path=WORKSPACE_ROOT / "data" / "youtube_episodes.json",
)
source_episodes = [
    episode
    for episode in media.episodes()
    if episode.key in states
    and Stage.CHARACTER_ASSIGNER in states[episode.key].completed_stages
]
folds = assign_folds(source_episodes)
splits = select_splits(folds, test_fold=0, validation_fold=1)

rows = []
quality_exclusions = {}
anonymous_segments_removed = 0
for episode in source_episodes:
    premise = premises.get(episode.key)
    if premise is None:
        continue

    episode.transcript = source_tracker.stage_transcript(
        states[episode.key].run_id,
        Stage.CHARACTER_ASSIGNER,
    )
    anonymous_segments_removed += sum(
        segment.get("speaker") == ANONYMOUS_LABEL
        for segment in episode.transcript.segments
    )
    try:
        pair = build_pair(episode, premise)
    except TranscriptQualityError as error:
        quality_exclusions[episode.key] = error.reasons
        continue
    rows.append({**pair, "split": splits[episode.key]})

if not rows:
    raise ValueError("No valid premise/dialogue pairs found")

print(f"valid pairs: {len(rows)}")
print(f"empty premises filtered: {len(empty_episode_keys)}")
print(f"quality exclusions: {len(quality_exclusions)}")
print(f"anonymous segments removed: {anonymous_segments_removed}")
if quality_exclusions:
    print(quality_exclusions)
print(
    "splits:",
    {
        split: sum(row["split"] == split for row in rows)
        for split in ("train", "validation", "test")
    },
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dataset_builder = SFTDatasetBuilder(
    tokenizer,
    max_length=MAX_LENGTH,
    enable_thinking=False,
)
split_rows, overlong_episode_keys = dataset_builder.prepare_rows(rows)
dataset = dataset_builder.build_dataset(split_rows)
training_rows = [
    row
    for split in ("train", "validation", "test")
    for row in split_rows[split]
]

print(dataset)
print(f"overlong episodes filtered: {overlong_episode_keys}")
print(
    "token range:",
    min(row["token_count"] for row in training_rows),
    max(row["token_count"] for row in training_rows),
)

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)
model = get_peft_model(
    model,
    LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "v_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    ),
)
model.print_trainable_parameters()

OUTPUT_DIR = mkdtemp(prefix=f"{MODEL_KEY}-qlora-")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    max_grad_norm=MAX_GRAD_NORM,
    optim="paged_adamw_8bit",
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=1,
    report_to="mlflow",
    run_name=RUN_NAME,
    seed=SEED,
    data_seed=SEED,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

first_training_row = split_rows["train"][0]
expected_prompt_ids = tokenizer.apply_chat_template(
    first_training_row["prompt"],
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
)["input_ids"]
expected_full_ids = tokenizer.apply_chat_template(
    first_training_row["prompt"] + first_training_row["completion"],
    tokenize=True,
    add_generation_prompt=False,
    enable_thinking=False,
    return_dict=True,
)["input_ids"]
if expected_full_ids[: len(expected_prompt_ids)] != expected_prompt_ids:
    raise RuntimeError("Tokenizer prompt mismatch before completion")

first_training_example = trainer.train_dataset[0]
input_ids = first_training_example["input_ids"]
labels = first_training_example.get("labels")
if labels is None:
    raise RuntimeError("TRL did not create completion-only labels")
masked_prompt_tokens = sum(label == -100 for label in labels)
supervised_completion_tokens = len(labels) - masked_prompt_tokens
if input_ids != expected_full_ids:
    raise RuntimeError("TRL tokenization differs from the expected chat template")
if masked_prompt_tokens != len(expected_prompt_ids):
    raise RuntimeError(
        f"Expected {len(expected_prompt_ids)} masked prompt tokens, "
        f"found {masked_prompt_tokens}"
    )
if labels[:masked_prompt_tokens] != [-100] * masked_prompt_tokens:
    raise RuntimeError("Prompt labels are not fully masked")
if labels[masked_prompt_tokens:] != input_ids[masked_prompt_tokens:]:
    raise RuntimeError("Completion labels do not match completion tokens")
print(
    "completion-only labels:",
    {
        "prompt_tokens": masked_prompt_tokens,
        "completion_tokens": supervised_completion_tokens,
    },
)

In [ ]:
mlflow.set_experiment(TRAINING_EXPERIMENT_NAME)
with mlflow.start_run(run_name=RUN_NAME) as run:
    TRAINING_RUN_ID = run.info.run_id
    mlflow.set_tags(
        {
            "model_key": MODEL_KEY,
            "model_name": MODEL_NAME,
            "training_method": "qlora",
            "dataset_paradigm": "full_episode",
            "source_run_id": SOURCE_RUN_ID,
            "premise_run_id": PREMISE_RUN_ID,
        }
    )
    mlflow.log_params(
        {
            "model_key": MODEL_KEY,
            "max_length": MAX_LENGTH,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "lora_target_modules": "q_proj,v_proj",
            "batch_size": BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "learning_rate": LEARNING_RATE,
            "epochs": TRAIN_EPOCHS,
            "warmup_ratio": WARMUP_RATIO,
            "max_grad_norm": MAX_GRAD_NORM,
            "transformers_version": transformers.__version__,
            "trl_version": trl.__version__,
            "peft_version": peft.__version__,
            "seed": SEED,
        }
    )
    mlflow.log_metrics(
        {
            "dataset.valid_pairs": len(rows),
            "dataset.empty_premises": len(empty_episode_keys),
            "dataset.quality_exclusions": len(quality_exclusions),
            "dataset.anonymous_segments_removed": anonymous_segments_removed,
            "dataset.overlong_examples": len(overlong_episode_keys),
            **{
                f"dataset.{split}_examples": len(items)
                for split, items in split_rows.items()
            },
        }
    )
    mlflow.log_dict(
        {
            "source_experiment_name": SOURCE_EXPERIMENT_NAME,
            "source_run_id": SOURCE_RUN_ID,
            "premise_experiment_name": PREMISE_EXPERIMENT_NAME,
            "premise_run_id": PREMISE_RUN_ID,
            "empty_episode_keys": empty_episode_keys,
            "quality_exclusions": quality_exclusions,
            "anonymous_segments_removed": anonymous_segments_removed,
            "overlong_episode_keys": overlong_episode_keys,
            "rows": [
                {
                    "episode_key": row["episode_key"],
                    "split": row["split"],
                    "premise": row["premise"],
                    "token_count": row["token_count"],
                }
                for row in training_rows
            ],
        },
        "dataset_manifest.json",
    )

    train_result = trainer.train()
    eval_metrics = trainer.evaluate()
    mlflow.log_metrics(
        {
            key: float(value)
            for key, value in eval_metrics.items()
            if isinstance(value, (int, float))
        }
    )

    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    mlflow.log_artifacts(OUTPUT_DIR, artifact_path="adapter")

print(f"training run: {TRAINING_RUN_ID}")
print(eval_metrics)

In [ ]:
heldout = split_rows["test"][10]
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

comparison = {
    "episode_key": heldout["episode_key"],
    "premise": heldout["premise"],
    "base": generate_episode(
        model,
        tokenizer,
        heldout["premise"],
        adapter_enabled=False,
        seed=SEED,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=GENERATION_DO_SAMPLE,
        temperature=GENERATION_TEMPERATURE,
        top_p=GENERATION_TOP_P,
        top_k=GENERATION_TOP_K,
        repetition_penalty=GENERATION_REPETITION_PENALTY,
    ),
    "qlora": generate_episode(
        model,
        tokenizer,
        heldout["premise"],
        adapter_enabled=True,
        seed=SEED,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=GENERATION_DO_SAMPLE,
        temperature=GENERATION_TEMPERATURE,
        top_p=GENERATION_TOP_P,
        top_k=GENERATION_TOP_K,
        repetition_penalty=GENERATION_REPETITION_PENALTY,
    ),
}
MlflowClient().log_dict(
    TRAINING_RUN_ID,
    comparison,
    "heldout_comparison.json",
)
print(comparison)